In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MongoDB_DataLake_Project")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [2]:
import os

from dotenv import load_dotenv
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv(r"C:\Professional_project\MongoDB_DataLake\credentials.env")

ACCOUNT_NAME = os.getenv("AZURE_STORAGE_ACCOUNT")
ACCOUNT_KEY = os.getenv("AZURE_STORAGE_KEY")

ACCOUNT_URL = f"https://{ACCOUNT_NAME}.dfs.core.windows.net"

service_client = DataLakeServiceClient(
    account_url=ACCOUNT_URL,
    credential=ACCOUNT_KEY
)

file_system_client = service_client.get_file_system_client("bronze")
directory_client = file_system_client.get_directory_client("mongodb")

files = list(directory_client.get_paths())

latest_file = max(
    files,
    key=lambda x: x.last_modified
)

print(latest_file.name)

mongodb/transactions_20260705_124610.json


In [3]:
download_path = r"C:\Professional_project\MongoDB_DataLake\temp\bronze\transactions.json"

file_client = file_system_client.get_file_client(latest_file.name)

download = file_client.download_file()

with open(download_path, "wb") as f:
    f.write(download.readall())

print("Download completed!")

Download completed!


In [4]:
df = spark.read.json(download_path)

df.printSchema()

df.show(5, truncate=False)

root
 |-- destination_account: struct (nullable = true)
 |    |-- account_id: string (nullable = true)
 |    |-- balance_after: double (nullable = true)
 |    |-- balance_before: double (nullable = true)
 |-- fraud: struct (nullable = true)
 |    |-- is_flagged: boolean (nullable = true)
 |    |-- is_fraud: boolean (nullable = true)
 |-- origin_account: struct (nullable = true)
 |    |-- account_id: string (nullable = true)
 |    |-- balance_after: double (nullable = true)
 |    |-- balance_before: double (nullable = true)
 |-- step: long (nullable = true)
 |-- transaction: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- type: string (nullable = true)

+------------------------------------+--------------+-------------------------------------+----+---------------------+
|destination_account                 |fraud         |origin_account                       |step|transaction          |
+------------------------------------+--------------+------------------

In [5]:
print(f"Rows: {df.count()}")

Rows: 190879


In [6]:
print(f"Columns: {len(df.columns)}")

Columns: 5


In [7]:
print(df.columns)

['destination_account', 'fraud', 'origin_account', 'step', 'transaction']


In [8]:
df.describe().show()

+-------+------------------+
|summary|              step|
+-------+------------------+
|  count|            190879|
|   mean|243.37764761969623|
| stddev|142.31425422641615|
|    min|                 1|
|    max|               741|
+-------+------------------+



In [9]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+-------------------+-----+--------------+----+-----------+
|destination_account|fraud|origin_account|step|transaction|
+-------------------+-----+--------------+----+-----------+
|                  0|    0|             0|   0|          0|
+-------------------+-----+--------------+----+-----------+



In [10]:
df.select("transaction.type").distinct().show()

+--------+
|    type|
+--------+
|TRANSFER|
| CASH_IN|
|CASH_OUT|
| PAYMENT|
|   DEBIT|
+--------+



In [11]:
df.groupBy("fraud.is_fraud").count().show()

+--------+------+
|is_fraud| count|
+--------+------+
|    true|   266|
|   false|190613|
+--------+------+



In [12]:
df_silver = df.select(
    col("step"),

    col("transaction.amount").alias("transaction_amount"),
    col("transaction.type").alias("transaction_type"),

    col("origin_account.account_id").alias("origin_account_id"),
    col("origin_account.balance_before").alias("origin_balance_before"),
    col("origin_account.balance_after").alias("origin_balance_after"),

    col("destination_account.account_id").alias("destination_account_id"),
    col("destination_account.balance_before").alias("destination_balance_before"),
    col("destination_account.balance_after").alias("destination_balance_after"),

    col("fraud.is_fraud").alias("is_fraud"),
    col("fraud.is_flagged").alias("is_flagged")
)

In [13]:
df_silver.printSchema()

root
 |-- step: long (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- origin_account_id: string (nullable = true)
 |-- origin_balance_before: double (nullable = true)
 |-- origin_balance_after: double (nullable = true)
 |-- destination_account_id: string (nullable = true)
 |-- destination_balance_before: double (nullable = true)
 |-- destination_balance_after: double (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- is_flagged: boolean (nullable = true)



In [14]:
df_silver.show(5, truncate=False)

+----+------------------+----------------+-----------------+---------------------+--------------------+----------------------+--------------------------+-------------------------+--------+----------+
|step|transaction_amount|transaction_type|origin_account_id|origin_balance_before|origin_balance_after|destination_account_id|destination_balance_before|destination_balance_after|is_fraud|is_flagged|
+----+------------------+----------------+-----------------+---------------------+--------------------+----------------------+--------------------------+-------------------------+--------+----------+
|8   |13380.99          |PAYMENT         |C1122673744      |0.0                  |0.0                 |M2018084215           |0.0                       |0.0                      |false   |false     |
|7   |2893.18           |PAYMENT         |C885018785       |123.0                |0.0                 |M2127513096           |0.0                       |0.0                      |false   |false     |


In [15]:
df_silver.rdd.getNumPartitions()

4

In [15]:
silver_path = r"C:\Professional_project\MongoDB_DataLake\temp\silver"

df_silver.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Parquet files created successfully!")

Parquet files created successfully!
